# llmon vs Ollama, side by side — Colab T4 edition

Installs **both** engines, serves the **same Qwen2.5-0.5B weights** in each,
and races them with identical prompts. Run cells top to bottom.
Takes ~15 min total (mostly compiling the CUDA backend once).

| engine | port |
|---|---|
| Ollama | `11434` |
| llmon | `11435` (default) |

In [ ]:
%%bash
nvidia-smi --query-gpu=name,memory.total --format=csv  # expect Tesla T4

## 1. Ollama (reference)

In [ ]:
%%bash
set -e
if ! command -v ollama >/dev/null; then
  curl -fsSL https://ollama.com/install.sh | sh
fi
export PATH="/usr/local/bin:$PATH"
pkill -f 'ollama serve' 2>/dev/null || true
nohup ollama serve >/tmp/ollama.log 2>&1 &
sleep 5
ollama pull qwen2.5:0.5b
ollama list

## 2. llmon + CUDA backend

Builds the Rust binary (~2 min) and the CUDA llama.cpp backend
(~10 min, T4 arch auto-detected as 75).

In [ ]:
%%bash
set -e
export PATH="$HOME/.cargo/bin:$HOME/.local/bin:$PATH"
if ! command -v cargo >/dev/null; then
  curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal
fi
test -d /content/llmon || git clone --depth 1 https://github.com/deepsuthar496/llmon /content/llmon
cd /content/llmon && cargo build --release --locked
ln -sf /content/llmon/target/release/llmon "$HOME/.local/bin/llmon" 2>/dev/null || {
  mkdir -p "$HOME/.local/bin" && ln -sf /content/llmon/target/release/llmon "$HOME/.local/bin/llmon";
}
sudo apt-get update -qq && sudo apt-get install -y -qq cmake g++ > /dev/null
cd /content/llmon && ./scripts/build-backend.sh --cuda
llmon bench --tokens 32

## 3. Same weights in llmon, start its server

In [ ]:
%%bash
set -e
export PATH="$HOME/.local/bin:$PATH" LD_LIBRARY_PATH="$HOME/.local/lib:$LD_LIBRARY_PATH"
llmon pull Qwen/Qwen2.5-0.5B-Instruct-GGUF/qwen2.5-0.5b-instruct-q4_k_m.gguf
llmon list
pkill -f 'llmon serve' 2>/dev/null || true
nohup llmon serve >/tmp/llmon.log 2>&1 &
sleep 3 && curl -s localhost:11435/health && echo

## 4. Race: same prompts, same 60-token budget, medians of 3

In [ ]:
import json, statistics, time, urllib.request

PROMPTS = [
    "Summarize the water cycle in one paragraph",
    "Explain what a prime number is in one paragraph",
    "Describe the solar system in one paragraph",
]
TARGETS = {
    "ollama": ("http://localhost:11434/api/generate", "qwen2.5:0.5b"),
    "llmon": ("http://localhost:11435/api/generate",
              "Qwen/Qwen2.5-0.5B-Instruct-GGUF/qwen2.5-0.5b-instruct-q4_k_m.gguf"),
}

def run_once(url, model, prompt):
    body = json.dumps({"model": model, "prompt": prompt, "stream": True,
                       "options": {"num_predict": 60, "temperature": 0}}).encode()
    req = urllib.request.Request(url, data=body, headers={"Content-Type": "application/json"})
    t0, first = time.monotonic(), None
    for line in urllib.request.urlopen(req, timeout=300):
        line = line.strip()
        if not line:
            continue
        try:
            c = json.loads(line)
        except ValueError:
            continue
        if first is None:
            first = time.monotonic() - t0
        if c.get("done"):
            break
    return first or 0, time.monotonic() - t0

for name, (url, model) in TARGETS.items():
    run_once(url, model, "warmup")
    ts, ss = [], []
    for p in PROMPTS:
        t, s = run_once(url, model, p)
        ts.append(t); ss.append(s)
        print(f"{name}: TTFT={t:.2f}s total={s:.2f}s", flush=True)
    print(f"{name} MEDIAN: TTFT={statistics.median(ts):.2f}s total={statistics.median(ss):.2f}s\n")

## 5. Report

Paste the two MEDIAN lines into a llmon GitHub issue with:
`nvidia-smi` output, and whether either side errored.
Expected on T4: both engines in the same ballpark (same kernels) —
llmon should match or beat Ollama on repeated prompts via ngram
speculative decoding. If it doesn't, that's a bug worth filing.